In [1]:
import pandas as pd
import numpy as np
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

In [2]:
df_sample = pd.read_csv('mid_project/sample_submission.csv')
df_test = pd.read_csv('mid_project/test_copy.csv')
df_train = pd.read_csv('mid_project/train_df.csv')

In [3]:
df_sample.shape, df_test.shape, df_train.shape

((48108, 2), (48108, 8), (50512, 9))

In [4]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48108 entries, 0 to 48107
Data columns (total 8 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   age                              48108 non-null  int64  
 1   number_dependent_family_members  48108 non-null  int64  
 2   monthly_income                   48108 non-null  float64
 3   number_of_credit_lines           48108 non-null  int64  
 4   real_estate_loans                48108 non-null  int64  
 5   ratio_debt_payment_to_income     48108 non-null  float64
 6   credit_line_utilization          48108 non-null  float64
 7   total_late_payments              48108 non-null  int64  
dtypes: float64(3), int64(5)
memory usage: 2.9 MB


In [5]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50512 entries, 0 to 50511
Data columns (total 9 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   age                              50512 non-null  int64  
 1   number_dependent_family_members  50512 non-null  int64  
 2   monthly_income                   50512 non-null  float64
 3   number_of_credit_lines           50512 non-null  int64  
 4   real_estate_loans                50512 non-null  int64  
 5   ratio_debt_payment_to_income     50512 non-null  float64
 6   credit_line_utilization          50512 non-null  float64
 7   total_late_payments              50512 non-null  int64  
 8   target                           50512 non-null  int64  
dtypes: float64(3), int64(6)
memory usage: 3.5 MB


In [6]:
X = df_train.drop(['target'], axis=1)
y = df_train['target']
X.shape, y.shape

((50512, 8), (50512,))

In [7]:
def objective(trial):

    params = {
        'scale_pos_weight': 13.4,
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': 4,
        'verbosity': -1,
        'n_jobs': -1
    }

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=5)
    scores = []

    for fold, (train_index, val_index) in enumerate(skf.split(X, y)):
        X_train, X_val = X.iloc[train_index], X.iloc[val_index]
        y_train, y_val = y.iloc[train_index], y.iloc[val_index]

        model = lgb.LGBMClassifier(**params)
        model.fit(X_train, y_train)
        y_pred = model.predict_proba(X_val)[:, 1]
        roc_auc = roc_auc_score(y_val, y_pred)
        scores.append(roc_auc)

    return np.mean(scores)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=40)

[I 2026-02-10 13:43:47,146] A new study created in memory with name: no-name-dc7e5180-2f87-4578-8550-b54d98280a6d
[I 2026-02-10 13:43:53,173] Trial 0 finished with value: 0.7910023180328632 and parameters: {'n_estimators': 768, 'learning_rate': 0.06629451731542574, 'max_depth': 11, 'num_leaves': 104, 'min_child_samples': 38, 'subsample': 0.9207579288005595, 'colsample_bytree': 0.767869453921368, 'reg_alpha': 1.1597202358688747e-08, 'reg_lambda': 8.902659730104674e-05}. Best is trial 0 with value: 0.7910023180328632.
[I 2026-02-10 13:43:57,424] Trial 1 finished with value: 0.7923579591634505 and parameters: {'n_estimators': 932, 'learning_rate': 0.10938756210046936, 'max_depth': 10, 'num_leaves': 56, 'min_child_samples': 92, 'subsample': 0.9626600951927928, 'colsample_bytree': 0.7277252723659311, 'reg_alpha': 0.017221230657084596, 'reg_lambda': 9.511744597418866e-06}. Best is trial 1 with value: 0.7923579591634505.
[I 2026-02-10 13:43:59,401] Trial 2 finished with value: 0.8374604082363

In [8]:
best_params = study.best_params
best_params

{'n_estimators': 138,
 'learning_rate': 0.03384075238072635,
 'max_depth': 3,
 'num_leaves': 21,
 'min_child_samples': 21,
 'subsample': 0.7170804631713709,
 'colsample_bytree': 0.5773473534692393,
 'reg_alpha': 0.000578440871859455,
 'reg_lambda': 0.17108202852244495}

In [9]:
print(f'ROC AUC score :{study.best_value:.4f}')

ROC AUC score :0.8416


In [10]:
lgb_model = lgb.LGBMClassifier(**best_params, random_state=4)

In [11]:
lgb_model.fit(X, y)

,boosting_type,'gbdt'
,num_leaves,21
,max_depth,3
,learning_rate,0.03384075238072635
,n_estimators,138
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,21


In [13]:
y_pred_lgb = lgb_model.predict_proba(df_test)

In [14]:
df_sample['Predicted'] = y_pred_lgb
df_sample.head()

,Id,Predicted
0,1,0.904482
1,2,0.545120
2,3,0.985716
3,4,0.975921
4,5,0.635226


In [ ]:
df_sample.to_csv('submission.csv', index=False)

In [15]:
# import joblib
# joblib.dump(lgb_model, 'lgb_kfold.joblib')

['lgb_kfold.joblib']